# 1kg_eur — round-2 ancestry gate

Refits PCA on the round-1 European participants, projects 1000G Europeans into
that space, and keeps participants whose Mahalanobis distance from the
projected CEU + GBR centroid falls within the 90th percentile of the anchor's
own distances.

**Why an anchor rather than the cohort's own centroid:** a self-centred gate
drifts with whoever is in the round-1 set. CEU + GBR (the Kemper et al.
anchor) fixes the centre and PC weighting to a population defined independently
of this cohort.

**Runs:** QC and PCA submitted as Google Batch jobs (dsub); scoring and gate
fitting run locally on the VM.

Round-1 keep list (prerequisite): produced by `01_round1_gate.ipynb`.

## config

In [ ]:
import os, sys, subprocess, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2

# ── paths ────────────────────────────────────────────────────────────────────
WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS  = f"{WS_GS}/1kg_eur"
KG_DIR = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot/1000g_reference")

ROUND1_KEEP = (f"{WS_GS}/phenotypic_covariance_v9"
               "/analyses/ancestry_filtering/keep/EUR_99pct_keep_ids.txt")
OUT_GS      = f"{R_GS}/01_ancestry/round2"
LOCAL       = os.path.expanduser("~/scratch_1kg_eur_gate")
os.makedirs(LOCAL, exist_ok=True)

# ── Batch config ──────────────────────────────────────────────────────────────
PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

# ── PCA parameters ────────────────────────────────────────────────────────────
N_PCS_FIT   = 20   # PCs fitted
K_PCS       = 5    # PCs used for the gate (set after inspecting pick-K output)
COVERAGE    = 0.90 # fraction of anchor samples the gate radius encloses

# ── reference populations ─────────────────────────────────────────────────────
ANCHOR_POPS = ["CEU", "GBR"]
EUR_POPS    = ["CEU", "GBR", "FIN", "TSI", "IBS"]
POP_COLORS  = {"CEU": "#e6194b", "GBR": "#f58231", "FIN": "#3cb44b",
               "TSI": "#4363d8", "IBS": "#911eb4"}

print(f"output: {OUT_GS}")
print(f"local scratch: {LOCAL}")

## install plink2 and dsub

plink2 is downloaded once and cached at `~/bin/plink2`.
dsub's cloud-sdk wrapper image is patched to a live tag — the default tag
in every dsub release is eventually withdrawn from gcr.io, which causes jobs
to fail silently at image pull.

In [ ]:
%%bash
# plink2
BIN_DIR="$HOME/bin"; mkdir -p "$BIN_DIR"
if [ ! -x "$BIN_DIR/plink2" ]; then
  cd /tmp
  wget -q -O plink2.zip     "https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR" && chmod +x "$BIN_DIR/plink2"
fi
export PATH="$HOME/bin:$PATH"
plink2 --version

# dsub (patch CLOUD_SDK_IMAGE in google_utils.py — must be in the same shell
# as submission; pip install reverts the patch)
pip install --quiet --upgrade 'dsub>=0.5.3'
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E   "s|cloud-sdk:[0-9]+\.[0-9]+\.[0-9]+-slim|cloud-sdk:581.0.0-slim|g"   "$DSUB_DIR/providers/google_utils.py"
echo "wrapper image: $(grep CLOUD_SDK_IMAGE "$DSUB_DIR/providers/google_utils.py")"
dsub --version

## long-range LD exclusion regions (GRCh38)

Written to a file for `plink2 --exclude bed1`. Includes the canonical list
plus narrow LD-peak intervals found by inspecting loadings from the first
gate PCA pass.

In [ ]:
LD_REGIONS_PATH = f"{LOCAL}/high_ld_regions.txt"
LD_REGIONS_GS   = f"{OUT_GS}/high_ld_regions.txt"

LD_REGIONS = """\
chr1 47761740 51761740 1
chr2 85919365 100517106 2
chr2 182427027 189427029 3
chr3 47483505 49987563 4
chr3 83368158 86868160 5
chr5 44464140 51168409 6
chr5 129636407 132636409 7
chr6 25391792 33424245 8
chr6 57788603 58453888 9
chr6 61109122 61357029 10
chr6 139637169 142137170 11
chr7 54964812 66897578 12
chr8 8105067 12105082 13
chr8 43025699 48924888 14
chr8 110918594 113918595 15
chr10 36671065 43184546 16
chr11 88127183 91127184 17
chr12 32955798 41319931 18
chr20 33948532 36438183 19
chr1 125169943 125170022 20
chr1 144106678 144106709 21
chr2 87416141 87417863 22
chr6 26726947 26726981 23
chr7 62182500 62277073 24
chr8 47303500 47317337 25
chr10 41693521 41885273 26
chr17 43159541 43159574 27
"""

with open(LD_REGIONS_PATH, "w") as fh:
    fh.write(LD_REGIONS)
subprocess.run(["gcloud", "storage", "cp", LD_REGIONS_PATH, LD_REGIONS_GS], check=True)
print(f"LD regions written ({LD_REGIONS.count(chr(10))-1} intervals)")

## QC, HM3 filter, and LD prune — Batch

One Batch job: restricts to round-1 participants, reapplies MAF/HWE/missingness
in that cohort, keeps variants agreeing with 1000G on ID and both alleles, then
prunes hard (r²=0.05 — the gate PCA needs clean axes, not coverage).

`r1_qc` is written to the bucket so `03_covariate_pca.ipynb` can start from
the same QC base without resubmitting.

In [ ]:
%%bash
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail

# patch CLOUD_SDK_IMAGE in same shell as submission
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E   "s|cloud-sdk:[0-9]+\.[0-9]+\.[0-9]+-slim|cloud-sdk:${CLOUD_SDK_TAG}|g"   "$DSUB_DIR/providers/google_utils.py"

PANEL_GS="${WS_GS}/genome_wide_panel/unified_panel_v9"
KEEP_GS="${ROUND1_KEEP}"
LD_GS="${OUT_GS}/high_ld_regions.txt"

dsub \
  --provider google-batch --project "$PROJECT_ID" --regions "$REGION" \
  --logging "${OUT_GS}/logs" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" --subnetwork "$SUBNETWORK" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:${CLOUD_SDK_TAG}" \
  --name "eur-r2-qc" \
  --machine-type "n1-highmem-16" --disk-size 600 \
  --input  PANEL_PGEN="${PANEL_GS}.pgen" \
  --input  PANEL_PVAR="${PANEL_GS}.pvar" \
  --input  PANEL_PSAM="${PANEL_GS}.psam" \
  --input  PLINK2_BIN="${WS_GS}/bin/plink2" \
  --input  KEEP_PATH="$KEEP_GS" \
  --input  LD_REGIONS="$LD_GS" \
  --output-recursive OUT_DIR="${OUT_GS}/panels" \
  --env    VCPUS=16 --env MEM_MB=112640 \
  --command '
    set -eo pipefail
    chmod +x "$PLINK2_BIN"
    Q="${OUT_DIR}/r1_qc"

    # QC in the round-1 cohort
    "$PLINK2_BIN" --pfile "$PANEL_PGEN_NO_EXT" --keep "$KEEP_PATH" --nonfounders \
      --maf 0.01 --hwe 1e-6 0 keep-fewhet --geno 0.05 \
      --max-alleles 2 --rm-dup exclude-all \
      --threads "$VCPUS" --memory "$MEM_MB" --make-pgen --out "$Q"

    # keep only variants matching 1000G on ID + REF + ALT
    grep -v "^##" "${Q}.pvar" | awk "NR>1 {print \$3, \$4, \$5}" | LC_ALL=C sort > /tmp/lhs
    awk "NR>1 {print \$2, \$3, \$4}" "${PANEL_PVAR_NO_EXT}_kg.acount" | LC_ALL=C sort > /tmp/kg
    LC_ALL=C comm -12 /tmp/lhs /tmp/kg | awk "{print \$1}" > "${OUT_DIR}/hm3.ids"

    # LD prune at r²=0.05 (tight — gate PCA needs clean axes)
    "$PLINK2_BIN" --pfile "$Q" --nonfounders \
      --extract "${OUT_DIR}/hm3.ids" \
      --exclude bed1 "$LD_REGIONS" \
      --indep-pairwise 1000kb 1 0.05 \
      --threads "$VCPUS" --out "${OUT_DIR}/prune"

    # pruned panel for PCA
    "$PLINK2_BIN" --pfile "$Q" --nonfounders \
      --extract "${OUT_DIR}/prune.prune.in" \
      --threads "$VCPUS" --memory "$MEM_MB" --make-pgen --out "${OUT_DIR}/pca_input"

    echo "r1_qc: $(wc -l < "${Q}.psam") samples"
    echo "after pruning: $(wc -l < "${OUT_DIR}/prune.prune.in") variants"
  ' 2>&1 | tee /tmp/qc_job.log
QC_JOB=$(tail -1 /tmp/qc_job.log)
echo "job id: $QC_JOB" | tee /tmp/qc_job_id.txt

In [ ]:
%%bash
# ALREADY RUN — remove the next line to resubmit
exit 0
# poll job status — re-run this cell to refresh
dstat --provider google-batch --project "$PROJECT_ID" \
  --location "$REGION" --jobs "$(cat /tmp/qc_job_id.txt)" \
  --users '*' --status '*' --full 2>&1 | tail -30

## PCA — Batch

Fit on the pruned round-1 panel. `--pca approx N allele-wts` keeps the allele
loadings so every cohort can be scored through the same weights.

In [ ]:
%%bash
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E "s|cloud-sdk:[0-9]+\.[0-9]+\.[0-9]+-slim|cloud-sdk:${CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "$PROJECT_ID" --regions "$REGION" \
  --logging "${OUT_GS}/logs" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" --subnetwork "$SUBNETWORK" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:${CLOUD_SDK_TAG}" \
  --name "eur-r2-pca" \
  --machine-type "n1-highmem-32" --disk-size 300 \
  --input  PGEN="${OUT_GS}/panels/pca_input.pgen" \
  --input  PVAR="${OUT_GS}/panels/pca_input.pvar" \
  --input  PSAM="${OUT_GS}/panels/pca_input.psam" \
  --input  PLINK2_BIN="${WS_GS}/bin/plink2" \
  --output-recursive OUT_DIR="${OUT_GS}/pca" \
  --env    VCPUS=32 --env MEM_MB=212992 --env N_PCS=20 \
  --command '
    set -eo pipefail
    chmod +x "$PLINK2_BIN"
    "$PLINK2_BIN" --pgen "$PGEN" --pvar "$PVAR" --psam "$PSAM" \
      --nonfounders --freq counts \
      --pca approx $N_PCS allele-wts \
      --threads "$VCPUS" --memory "$MEM_MB" --out "${OUT_DIR}/round2_pca"
    echo done
  ' 2>&1 | tee /tmp/pca_job.log
PCA_JOB=$(tail -1 /tmp/pca_job.log)
echo "job id: $PCA_JOB" | tee /tmp/pca_job_id.txt

In [ ]:
%%bash
# ALREADY RUN — remove the next line to resubmit
exit 0
dstat --provider google-batch --project "$PROJECT_ID" \
  --location "$REGION" --jobs "$(cat /tmp/pca_job_id.txt)" \
  --users '*' --status '*' --full 2>&1 | tail -30

## score everyone through the same loadings

Participants and 1000G Europeans are both scored through the PCA's allele
weights using `plink2 --score`. This is deliberate: `--pca` eigenvectors and
a `--score` projection land on different coordinates (the eigenvector is
normalised by the cohort's allele frequency; the projection uses the stored
weights directly). Re-scoring puts both sets on identical axes so Mahalanobis
distances are comparable.

In [ ]:
# download PCA outputs
subprocess.run(["bash", "-c", f"""
gsutil -m cp \\
  "{OUT_GS}/pca/round2_pca.eigenval" \\
  "{OUT_GS}/pca/round2_pca.eigenvec.allele" \\
  "{OUT_GS}/pca/round2_pca.acount" \\
  "{OUT_GS}/panels/pca_input.pvar" \\
  "{OUT_GS}/panels/prune.prune.in" \\
  "{LOCAL}/"
"""], check=True)

PCA    = f"{LOCAL}/round2_pca"
PANEL  = f"{LOCAL}/pca_input"
W      = f"{PCA}.eigenvec.allele"   # allele loadings
FREQ   = f"{PCA}.acount"            # allele counts — used as frequency reference
KG_BFILE = os.path.join(KG_DIR, "1kg_all_qc")
KG_ACOUNT = f"{KG_BFILE}.acount"

# shared variants: ID + REF + ALT match against 1000G
def shared_variants(pvar_or_alleles, out_ids, cols="3,4,5"):
    c = [f"${x}" for x in cols.split(",")]
    subprocess.run(["bash", "-c", f"""
grep -v '^##' "{pvar_or_alleles}" | awk 'NR>1 {{print {", ".join(c)}}}' \\
  | LC_ALL=C sort > "{out_ids}.lhs"
awk 'NR>1 {{print $2, $3, $4}}' "{KG_ACOUNT}" | LC_ALL=C sort > "{out_ids}.kg"
LC_ALL=C comm -12 "{out_ids}.lhs" "{out_ids}.kg" | awk '{{print $1}}' > "{out_ids}"
echo "shared with 1000G: $(wc -l < "{out_ids}") variants"
"""], check=True)
    return out_ids

kg_ids = shared_variants(f"{PANEL}.pvar", f"{LOCAL}/kg_in_r2.ids")

In [ ]:
# parse score column numbers from the weights header
h = open(W).readline().split()
idc = "#ID" if "#ID" in h else "ID"
_i, _a1 = h.index(idc) + 1, h.index("A1") + 1
_p1, _pk = h.index("PC1") + 1, h.index(f"PC{N_PCS_FIT}") + 1

def plink_score(weights, out, src_flag, freq, extract=None, n_pcs=N_PCS_FIT):
    e = f'--extract "{extract}"' if extract else ""
    subprocess.run(["bash", "-c", f"""
plink2 {src_flag} --nonfounders {e} \\
  --read-freq "{freq}" \\
  --score "{weights}" {_i} {_a1} header-read no-mean-imputation variance-standardize \\
  --score-col-nums {_p1}-{_pk} \\
  --threads $(nproc) --out "{out}"
echo "scored: $(($(wc -l < "{out}.sscore") - 1)) samples"
"""], check=True)

plink_score(W, f"{LOCAL}/kg_scores",
            f'--bfile "{KG_BFILE}"', FREQ, extract=kg_ids)
plink_score(W, f"{LOCAL}/part_scores",
            f'--pfile "{PANEL}"', FREQ)

In [ ]:
# read scores into dataframes
def read_scores(sscore, id_col, n_pcs=N_PCS_FIT):
    d = pd.read_csv(sscore, sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: id_col,
                           **{f"PC{k}_AVG": f"PC{k}" for k in range(1, n_pcs+1)}})
    d[id_col] = d[id_col].astype(str)
    return d[[id_col] + [f"PC{k}" for k in range(1, n_pcs+1)]]

kg_panel = pd.read_csv(
    os.path.join(KG_DIR, "integrated_call_samples_v3.20130502.ALL.panel"),
    sep=r"\s+")[["sample","pop","super_pop"]]

kg   = read_scores(f"{LOCAL}/kg_scores.sscore", "sample").merge(kg_panel, on="sample", how="left")
part = read_scores(f"{LOCAL}/part_scores.sscore", "person_id")

ev   = np.loadtxt(f"{PCA}.eigenval")
pct  = ev / ev.sum() * 100
print(f"{len(part):,} participants scored")
print("  ".join(f"PC{k+1} {pct[k]:.2f}%" for k in range(10)))

## pick K

Print separation of CEU+GBR from the other EUR populations in participant SDs
for each PC. Use PCs where the gap is large; a PC that does not separate the
anchor adds noise to the Mahalanobis distance without adding selectivity.
Set `K_PCS` in the config cell above, then re-run from the gate cell.

In [ ]:
PC = [f"PC{k}" for k in range(1, N_PCS_FIT + 1)]
anchor = kg[kg["pop"].isin(ANCHOR_POPS)]
other_eur = kg[kg["pop"].isin(EUR_POPS) & ~kg["pop"].isin(ANCHOR_POPS)]

rows = []
for pc in PC[:10]:
    mu_a = anchor[pc].mean();  sd_p = part[pc].std()
    mu_o = other_eur[pc].mean()
    rows.append({"PC": pc, "pct_var": round(pct[int(pc[2:])-1], 2),
                 "anchor_vs_other_EUR_SDs": round(abs(mu_a - mu_o) / sd_p, 3)})
print(pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (i, j) in zip(axes, [(1,2),(3,4)]):
    a, b = f"PC{i}", f"PC{j}"
    rng  = np.random.default_rng(42)
    idx  = rng.choice(len(part), size=min(30_000, len(part)), replace=False)
    ax.scatter(part[a].iloc[idx], part[b].iloc[idx], s=1, alpha=0.1, color="0.8", rasterized=True)
    for pop in EUR_POPS:
        s = kg[kg["pop"]==pop]
        ax.scatter(s[a], s[b], s=30, marker="x", color=POP_COLORS[pop], label=pop, zorder=3)
    ax.set_xlabel(a); ax.set_ylabel(b)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{LOCAL}/pick_k.png", dpi=150)
plt.show()

## loadings check

Plot PC loadings by chromosomal position. A narrow peak concentrated on one
chromosome region means a single locus is driving that PC. If any are found,
add the region to the LD exclusion list and rerun from the QC job.

In [ ]:
L = pd.read_csv(f"{PCA}.eigenvec.allele", sep=r"\s+")
idc = "#ID" if "#ID" in L.columns else "ID"
L[["CHROM","POS"]] = L[idc].str.split(":", n=2, expand=True).iloc[:,:2]
L["POS"] = L["POS"].astype(int)

fig, axes = plt.subplots(K_PCS, 1, figsize=(14, 2.5 * K_PCS), sharex=False)
for k, ax in enumerate(axes, 1):
    col = f"PC{k}"
    ax.scatter(range(len(L)), L[col].abs(), s=0.3, alpha=0.4, rasterized=True)
    ax.set_ylabel(col)
    ax.set_title(f"{col} loadings by position")
plt.tight_layout()
plt.savefig(f"{LOCAL}/loadings.png", dpi=150)
plt.show()
print("If any PC shows a sharp spike, add the genomic interval to the LD_REGIONS list above.")

## the gate

Mahalanobis distance under a multivariate normal fitted to the projected
CEU + GBR anchor. The gate radius is set to the `COVERAGE` quantile of the
anchor's own distances, so it encloses exactly that fraction of anchor samples.

Projection shrinks anchor scores toward the origin, but the coverage quantile
absorbs that rescaling — the anchor and participants are on the same axes.

In [ ]:
PC_GATE = [f"PC{k}" for k in range(1, K_PCS + 1)]

A = anchor[PC_GATE].to_numpy()
mu  = A.mean(0)
cov = np.cov(A, rowvar=False)
cov_inv = np.linalg.inv(cov)

def mahal(X, mu, cov_inv):
    d = X - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", d, cov_inv, d))

d_anchor = mahal(A, mu, cov_inv)
threshold = float(np.quantile(d_anchor, COVERAGE))
print(f"anchor: {len(A)} samples")
print(f"threshold ({COVERAGE:.0%} of anchor): {threshold:.4f}")

d_part = mahal(part[PC_GATE].to_numpy(), mu, cov_inv)
keep   = d_part <= threshold
print(f"kept: {keep.sum():,} / {len(part):,} ({keep.mean():.1%})")

In [ ]:
KEEP_PATH = f"{LOCAL}/1kg_eur_keep_ids.txt"
part.loc[keep, "person_id"].to_csv(KEEP_PATH, index=False, header=False)
subprocess.run(["gcloud", "storage", "cp", KEEP_PATH, f"{OUT_GS}/1kg_eur_keep_ids.txt"], check=True)
print(f"keep list written: {keep.sum():,} participants")

In [ ]:
# scatter: kept (blue) vs excluded (grey), 1000G EUR (crosses)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
rng = np.random.default_rng(42)
for ax, (i, j) in zip(axes, [(1,2),(3,4)]):
    a, b = f"PC{i}", f"PC{j}"
    excl_idx = rng.choice(np.flatnonzero(~keep), size=min(20_000, (~keep).sum()), replace=False)
    kept_idx = rng.choice(np.flatnonzero(keep),  size=min(50_000, keep.sum()), replace=False)
    ax.scatter(part[a].iloc[excl_idx], part[b].iloc[excl_idx], s=1, alpha=0.15, color="0.8", rasterized=True)
    ax.scatter(part[a].iloc[kept_idx], part[b].iloc[kept_idx], s=1, alpha=0.2, color="tab:blue", rasterized=True, label="kept")
    for pop in EUR_POPS:
        s = kg[kg["pop"]==pop]
        ax.scatter(s[a], s[b], s=40, marker="x", color=POP_COLORS[pop], label=pop, zorder=4)
    ax.set_xlabel(a); ax.set_ylabel(b)
axes[0].legend(fontsize=8)
plt.suptitle(f"1kg_eur gate — {keep.sum():,} kept (blue), {(~keep).sum():,} excluded (grey)")
plt.tight_layout()
plt.savefig(f"{LOCAL}/round2_gate.png", dpi=150)
plt.show()

## comparison gates

The two alternatives bound how much the anchor choice matters:
- **self-centred**: participants' own centroid (what eur_D2 used — no external reference)
- **Kemper direction**: PCA fit on 1000G Europeans, participants projected in

In [ ]:
# self-centred gate
P  = part[PC_GATE].to_numpy()
mu_s = P.mean(0); cov_s = np.cov(P, rowvar=False)
d_self = mahal(P, mu_s, np.linalg.inv(cov_s))
thresh_self = float(np.quantile(d_self, COVERAGE))
keep_self = d_self <= thresh_self
print(f"self-centred: {keep_self.sum():,} kept")
print(f"  only in anchor gate:  {(keep & ~keep_self).sum():,}")
print(f"  only in self gate:    {(keep_self & ~keep).sum():,}")
print(f"  in both:              {(keep & keep_self).sum():,}")

Next: `03_covariate_pca.ipynb`